# 05 - Load the Bronze layer (SQL Server)

The database follows a **3-layer (medallion) architecture**:

| Layer | Content |
|---|---|
| `bronze` | Raw data exactly as collected (Kaggle files + scraped Inside Airbnb listings) |
| `silver` | Cleaned and enriched data |
| `gold` | Normalized tables with primary/foreign keys, ready for Power BI |

This notebook loads the **raw** data into `bronze` using **pandas + SQLAlchemy**.
Nothing is cleaned here; only two load-tracking columns are added (`_source`, `_loaded_at`).

In [1]:
from pathlib import Path

import pandas as pd
import pyodbc
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from sqlalchemy.types import NVARCHAR

PROCESSED = Path("../data/processed")
EXTERNAL = Path("../data/external/insideairbnb")

## Step 1: Connect to SQL Server
- Python talks to SQL Server through an **ODBC driver**. We pick the newest one installed.
- `SERVER=.` means "this computer", the same server name used in SSMS.
- `Trusted_Connection=yes` uses your Windows login (Windows Authentication), so no password is needed.
- `fast_executemany=True` sends rows in large batches, which makes loading much faster.

In [2]:
drivers = [d for d in pyodbc.drivers() if "ODBC Driver" in d and "SQL Server" in d]
print("SQL Server ODBC drivers found:", drivers)
if not drivers:
    raise RuntimeError("No ODBC driver found. Install 'ODBC Driver 18 for SQL Server' from Microsoft, then restart the kernel.")
DRIVER = sorted(drivers)[-1]

conn_str = (
    f"DRIVER={{{DRIVER}}};"
    "SERVER=.;"
    "DATABASE=AirbnbEurope;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)
engine = create_engine(URL.create("mssql+pyodbc", query={"odbc_connect": conn_str}), fast_executemany=True)

with engine.connect() as conn:
    print("Connected to:", conn.execute(text("SELECT DB_NAME()")).scalar())
    print(conn.execute(text("SELECT @@VERSION")).scalar().splitlines()[0])

SQL Server ODBC drivers found: ['ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server']
Connected to: AirbnbEurope
Microsoft SQL Server 2025 (RTM) - 17.0.1000.7 (X64) 


## Step 2: Prepare the Kaggle data
This is the merged file from notebook 01, still uncleaned.
The only change is renaming `Unnamed: 0` to `source_row_id`, because a column name with a colon and a space is awkward to use in SQL.

In [3]:
loaded_at = pd.Timestamp.now().floor("s")

kaggle = pd.read_csv(PROCESSED / "airbnb_master_raw.csv").rename(columns={"Unnamed: 0": "source_row_id"})
kaggle["_source"] = "kaggle"
kaggle["_loaded_at"] = loaded_at

print(kaggle.shape)

(51707, 24)


## Step 3: Prepare the scraped Inside Airbnb listings
The full files have about 90 columns. We keep the columns relevant to this project and store them as they are
(for example `price` is still text like `$120.00`, and `host_is_superhost` is still `t` / `f`).

In [4]:
IAB_COLS = [
    "id", "neighbourhood_cleansed", "latitude", "longitude", "property_type", "room_type",
    "accommodates", "bedrooms", "amenities", "number_of_reviews", "review_scores_rating",
    "host_id", "host_is_superhost", "host_listings_count", "availability_365", "price",
]

frames = []
for row in pd.read_csv(EXTERNAL / "sources.csv").itertuples():
    if row.file != "data/listings.csv.gz":
        continue
    listings = pd.read_csv(row.local_path, usecols=lambda c: c in IAB_COLS, low_memory=False)
    listings.insert(0, "city", row.city.title())
    listings.insert(1, "snapshot_date", row.snapshot_date)
    frames.append(listings)
    print(f"{row.city:<10} {len(listings):>6} listings")

iab = pd.concat(frames, ignore_index=True)
iab["_source"] = "insideairbnb.com (CC BY 4.0)"
iab["_loaded_at"] = loaded_at

print("\nTotal:", iab.shape)

amsterdam   10612 listings
athens      15134 listings
barcelona   15236 listings
berlin      12376 listings
budapest    11240 listings
lisbon      25020 listings
london      91954 listings
paris       77149 listings
rome        37451 listings
vienna      12982 listings

Total: (309154, 20)


## Step 4: Write both tables to the `bronze` schema
- Text columns are stored as **NVARCHAR** (Unicode). The default VARCHAR type cannot store Greek or some Hungarian letters
  and silently replaces them with `?`, which would break neighbourhood names in Athens and Budapest.
- `if_exists="replace"` drops and recreates the table, so this notebook can be re-run safely.
- `chunksize` sends the rows in groups of 5,000 instead of all at once.
- The Inside Airbnb table is large, so this step can take a few minutes.

In [5]:
def unicode_text(frame):
    """Map every text column to NVARCHAR(max) so no characters are lost."""
    return {
        col: NVARCHAR(None)
        for col in frame.columns
        if pd.api.types.is_object_dtype(frame[col]) or pd.api.types.is_string_dtype(frame[col])
    }


kaggle.to_sql("kaggle_listings", engine, schema="bronze", if_exists="replace", index=False,
              chunksize=5000, dtype=unicode_text(kaggle))
print("bronze.kaggle_listings done")

iab.to_sql("iab_listings", engine, schema="bronze", if_exists="replace", index=False,
           chunksize=5000, dtype=unicode_text(iab))
print("bronze.iab_listings done")

bronze.kaggle_listings done
bronze.iab_listings done


## Step 5: Verify the load
The number of rows in SQL Server must match the number of rows in pandas.

In [6]:
checks = {"bronze.kaggle_listings": len(kaggle), "bronze.iab_listings": len(iab)}

with engine.connect() as conn:
    for table, expected in checks.items():
        in_sql = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        print(f"{table:<25} pandas: {expected:>7}   SQL: {in_sql:>7}   match: {expected == in_sql}")

bronze.kaggle_listings    pandas:   51707   SQL:   51707   match: True
bronze.iab_listings       pandas:  309154   SQL:  309154   match: True
